# Análise de Dados de Backtesting

Este notebook serve como base para realizar as análises do seu operacional.

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from db_manager import load_from_sqlite_to_pandas

# Configurações visuais
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)

In [12]:
# 1. Carregar os dados do banco de dados
df = load_from_sqlite_to_pandas()

print(f"Linhas carregadas: {len(df)}")
df.head(5)

Successfully loaded 19868 rows from trading_data
Linhas carregadas: 19868


,Data,Opem,Max,Min,Close,MME52,VWAP,Contador,StopATR3,LinhaQuant,OBV,OBVMME52,OBVMME200
0,2026-01-05 09:03:00,163160,163310,163065,163220,162825.61,163178.92,1,162803.00,162950.5,12427.18,12344.98,12325.95
1,2026-01-05 09:04:00,163225,163285,163150,163250,162841.63,163191.97,2,162952.25,162950.5,12471.65,12349.76,12327.40
2,2026-01-05 09:05:00,163255,163530,163225,163470,162865.34,163287.66,3,162968.00,162950.5,12582.84,12358.55,12329.94
3,2026-01-05 09:06:00,163470,163470,163305,163390,162885.14,163307.90,4,163148.25,162950.5,12516.97,12364.53,12331.80
4,2026-01-05 09:07:00,163390,163395,163280,163320,162901.55,163309.99,5,163148.25,162950.5,12482.17,12368.97,12333.30


In [15]:

from db_manager import load_from_sqlite_to_pandas
# 1. Carregar os dados do banco
df = load_from_sqlite_to_pandas()
df = df.sort_values('Data').reset_index(drop=True)
# 2. Criar a coluna SQD (Sinal de Quant Direção) usando .loc do Pandas
# "C" se Close > LinhaQuant (Comprador)
# "V" se Close < LinhaQuant (Vendedor)
df['SQD'] = ''  # Inicializa a coluna com strings vazias
df.loc[df['Close'] > df['LinhaQuant'], 'SQD'] = 'C'
df.loc[df['Close'] < df['LinhaQuant'], 'SQD'] = 'V'
# Visualizar o resultado
print("Contagem de sinais SQD:")
print(df['SQD'].value_counts())
df[['Data', 'Close', 'LinhaQuant', 'SQD']].head(10)


Successfully loaded 19868 rows from trading_data
Contagem de sinais SQD:
SQD
C    10605
V     9252
        11
Name: count, dtype: int64


,Data,Close,LinhaQuant,SQD
0,2026-01-05 09:03:00,163220,162950.5,C
1,2026-01-05 09:04:00,163250,162950.5,C
2,2026-01-05 09:05:00,163470,162950.5,C
3,2026-01-05 09:06:00,163390,162950.5,C
4,2026-01-05 09:07:00,163320,162950.5,C
5,2026-01-05 09:08:00,163260,162950.5,C
6,2026-01-05 09:09:00,163105,162950.5,C
7,2026-01-05 09:10:00,163075,162950.5,C
8,2026-01-05 09:11:00,163025,162950.5,C
9,2026-01-05 09:12:00,163065,162950.5,C


In [16]:
# 3. Criar a coluna de Sinal (Sinal_Crossover)
# Identifica quando o SQD muda de valor
# Venda para Compra ('V' -> 'C') = 1
# Compra para Venda ('C' -> 'V') = -1
df['Sinal'] = 0  # Inicializa sem sinal
# Condição: mudou o SQD e o anterior não era vazio
# Usamos shift() para comparar com a linha anterior
mudou_para_compra = (df['SQD'] == 'C') & (df['SQD'].shift(1) == 'V')
mudou_para_venda = (df['SQD'] == 'V') & (df['SQD'].shift(1) == 'C')
df.loc[mudou_para_compra, 'Sinal'] = 1
df.loc[mudou_para_venda, 'Sinal'] = -1
# Visualizar apenas onde o sinal foi gerado
print("Contagem de Sinais Gerados:")
print(df['Sinal'].value_counts())
# Mostrar os primeiros 10 sinais encontrados
df[df['Sinal'] != 0][['Data', 'Close', 'LinhaQuant', 'SQD', 'Sinal']].head(10)

Contagem de Sinais Gerados:
Sinal
 0    19371
 1      249
-1      248
Name: count, dtype: int64


,Data,Close,LinhaQuant,SQD,Sinal
11,2026-01-05 09:14:00,162945,163332.0,V,-1
17,2026-01-05 09:20:00,163370,162859.5,C,1
42,2026-01-05 09:45:00,163260,163505.0,V,-1
97,2026-01-05 10:40:00,162960,162642.0,C,1
155,2026-01-05 11:38:00,163135,163416.5,V,-1
189,2026-01-05 12:12:00,163205,162927.0,C,1
243,2026-01-05 13:06:00,163705,163907.5,V,-1
272,2026-01-05 13:35:00,163930,163822.5,C,1
443,2026-01-05 16:26:00,164415,164524.5,V,-1
507,2026-01-05 17:30:00,164340,164215.0,C,1


In [20]:
# 3. Rotina de Simulação
def simular_operacional(df):
    operacoes = []
    idx_sinais = df[df['Sinal'] == 1].index

    for i in idx_sinais:
        data_sinal = df.loc[i, 'Data']
        cont_sinal = df.loc[i, 'Contador']
        preco_entrada_alvo = df.loc[i, 'Max'] + 5

        # Verificar se existe candle posterior
        if i + 1 >= len(df):
            continue

        prox_candle = df.iloc[i + 1]

        # Critério de Entrada: Mínima do próximo candle >= ponto de entrada
        if prox_candle['Min'] >= preco_entrada_alvo:
            cont_entrada = prox_candle['Contador']
            preco_entrada_efetivo = preco_entrada_alvo

            # Busca pelo Stop nos candles seguintes
            cont_stop = None
            preco_stop = None

            for j in range(i + 1, len(df)):
                candle_atual = df.iloc[j]

                # Se Mínima < LinhaQuant, aciona stop
                if candle_atual['Min'] < candle_atual['LinhaQuant']:
                    cont_stop = candle_atual['Contador']
                    preco_stop = candle_atual['LinhaQuant']
                    break

            if cont_stop is not None:
                operacoes.append({
                    'Data do Sinal': data_sinal,
                    'Contador Sinal': cont_sinal,
                    'Contador Entrada': cont_entrada,
                    'Preço Entrada': preco_entrada_efetivo,
                    'Contador Stop': cont_stop,
                    'Preço Stop': preco_stop
                })
    return pd.DataFrame(operacoes)
# 4. Executar e exibir resultados
df_operacoes = simular_operacional(df)
print(f"Total de operações realizadas: {len(df_operacoes)}")
df_operacoes.head(20)

Total de operações realizadas: 1


,Data do Sinal,Contador Sinal,Contador Entrada,Preço Entrada,Contador Stop,Preço Stop
0,2026-01-05 18:24:00,562,1,164425,28,165156.5
